## Setup

In [1]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import contextily as ctx
import xyzservices as xyz
import matplotlib.pyplot as plt

# import kml reading and set supported driver
import fiona
fiona.drvsupport.supported_drivers["KML"] = "rw"

In [3]:
from gridsample.utils import create_ids, save_shapefiles

In [4]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
PROCESSED_DATA_DIR = DATA_DIR / "01_processed" / "Colonies"

## 1. Load colony shapes

In [5]:
colony_gdf = gpd.read_file(
    RAW_DATA_DIR / "colony_shapes" / "colony_shapes.kml",
    crs="4326",
    driver="KML",
).drop(columns=["Description"])

In [6]:
# remove z-dimension from shapes
colony_gdf.geometry = colony_gdf.geometry.apply(lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2)))

In [ ]:
colony_gdf

## 2. Load rooftops

In [8]:
from s2cell.s2cell import lat_lon_to_cell_id
import boto3

### Download rooftop data

Get the ID of the level 6 S2 Cell that this colony sits inside

In [ ]:
for i, row in colony_gdf.iterrows():
    lat = row.geometry.centroid.y
    lon = row.geometry.centroid.x
    s2_cell_id = lat_lon_to_cell_id(lat=lat, lon=lon, level=6)
    print(f"lat: {lat}, lon: {lon}, s2_cell_id: {s2_cell_id}")
    
s2_cell_id


Download closest S2 cell shapefile from https://beta.source.coop/vida/google-microsoft-open-buildings/geoparquet/by_country_s2/country_iso=IND/

In [ ]:
s2_rooftops_path = RAW_DATA_DIR / "rooftops" / f"{s2_cell_id}.parquet"

if s2_rooftops_path.exists():
    print("File already exists")
else:
    s3 = boto3.client("s3", endpoint_url="https://data.source.coop")
    s3.download_file(
        "vida",
        f"google-microsoft-open-buildings/geoparquet/by_country_s2/country_iso=IND/{s2_cell_id}.parquet",
        str(s2_rooftops_path),
    )
    print("File downloaded.")

### Load and process rooftop data

In [11]:
rooftop_gdf = gpd.read_parquet(s2_rooftops_path)
# rooftop_gdf.sample(10000).plot(edgecolor="k")

In [12]:
rooftop_gdf = rooftop_gdf.drop(columns=["boundary_id", "s2_id", "geohash", "bbox", "country_iso"])

In [13]:
rooftop_gdf["rooftop_id"] = create_ids(len(rooftop_gdf), f"ROOFTOP_S2_{s2_cell_id}_")

In [ ]:
rooftop_gdf

## 3. Match rooftops to colonies

In [ ]:
subset_rooftops_gdf = rooftop_gdf.sjoin(colony_gdf, how="inner", predicate="intersects").drop(columns=["index_right"])
subset_rooftops_gdf

### Produce plots and save rooftops to file per colony (to be edited manually on MyMaps)

In [16]:
for colony in colony_gdf["Name"].unique():
    folder_path = PROCESSED_DATA_DIR / colony
    folder_path.mkdir(parents=True, exist_ok=True)

    colony_rooftops = subset_rooftops_gdf[subset_rooftops_gdf["Name"] == colony]
    ax = colony_rooftops.plot(figsize=(10, 15))
    # ctx.add_basemap(ax, crs=subset_rooftops_gdf.crs.to_string(), source=xyz.TileProvider.from_qms("Google Satellite Hybrid"))
    ax.set_axis_off()
    plt.title(f"{colony} Rooftops")
    colony_gdf[colony_gdf["Name"] == colony].boundary.plot(ax=ax, color="k")
    plt.savefig(folder_path / "raw_rooftops.png", bbox_inches="tight")
    plt.close()

    save_shapefiles(
        colony_rooftops,
        folder_path,
        f"raw {colony} rooftops",
        formats=["parquet", "kml"],
    )

    ## Get rooftop stats
    colony_rooftops["area_in_meters"].hist(bins=50)
    plt.xlabel("Rooftop area (m2)")
    plt.ylabel("Number of rooftops")
    plt.title(f"{colony} Rooftop Area Distribution")
    plt.savefig(folder_path / "raw_rooftop_area_distribution.png")
    plt.close()

At this point, upload the per-colony rooftops KML to Google MyMaps and edit and correct the shapes, then download the edited KMLs and continue here.

## 4. Load edited rooftops and run analysis

### Load edited rooftop shape KMLs

In [23]:
temp_colony_rooftops_gdf_list = []
for filename in [
    "Crystal_Green_Welfare_Society_rooftops.kml",
    "Golden_City_rooftops.kml",
    "Golden_City_Prima_rooftops.kml",
    "Paras_Hermitage_Society_rooftops.kml",
    "Eco_Green_City_rooftops.kml"
]:
    temp_colony_rooftops_gdf = gpd.read_file(
        RAW_DATA_DIR / "mymaps_edited_colony_rooftops" / filename,
        crs="4326",
        driver="KML",
    ).drop(columns=["Description"])

    temp_colony_rooftops_gdf_list.append(temp_colony_rooftops_gdf)

edited_colony_rooftops_gdf = pd.concat(temp_colony_rooftops_gdf_list, ignore_index=True)
edited_colony_rooftops_gdf.geometry = edited_colony_rooftops_gdf.geometry.apply(
    lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2))
)

In [24]:
edited_rooftop_areas = edited_colony_rooftops_gdf.to_crs("24378").area
edited_colony_rooftops_gdf["rooftop_area_m2"] = edited_rooftop_areas
edited_colony_rooftops_gdf["rooftop_id"] = create_ids(len(edited_colony_rooftops_gdf), "ROOFTOP_")

In [25]:
# CANCELLED # divide area by 2 for Eco Green City and Crystal Green Welfare Society (since the shapes are for 2 neighbouring buildings and building count will be doubled too)
# edited_colony_rooftops_gdf.loc[
#     edited_colony_rooftops_gdf["Name"].isin(
#         ["Eco Green City", "Crystal Green Welfare Society"]
#     ),
#     "rooftop_area_m2",
# ] /= 2

In [ ]:
edited_colony_rooftops_gdf

### Run stats

In [27]:
edited_colony_rooftops_gdf["rooftop_area_category"] = pd.cut(
    edited_colony_rooftops_gdf["rooftop_area_m2"],
    bins=[
        0,
        20,
        50,
        100,
        200,
        np.inf,
    ],
    labels=[
        "Rooftop count <20m2",
        "Rooftop count 20-50m2",
        "Rooftop count 50-100m2",
        "Rooftop count 100-200m2",
        "Rooftop count >200m2",
    ],
)

### Save new plots and shapefiles

In [22]:
for colony in [
    "Golden City",
    "Golden City Prima",
    "Paras Hermitage Society",
    "Eco Green City",
    "Crystal Green Welfare Society",
]:
    folder_path = PROCESSED_DATA_DIR / colony
    folder_path.mkdir(parents=True, exist_ok=True)

    colony_rooftops = edited_colony_rooftops_gdf[
        edited_colony_rooftops_gdf["Name"] == colony
    ]
    ax = colony_rooftops.plot(figsize=(10, 15))
    ax.set_axis_off()
    plt.title(f"{colony} Rooftops")
    colony_gdf[colony_gdf["Name"] == colony].boundary.plot(ax=ax, color="k")
    plt.savefig(folder_path / "edited_colony_rooftops.png", bbox_inches="tight")
    plt.close()

    save_shapefiles(
        colony_rooftops,
        folder_path,
        f"edited {colony} rooftops",
        formats=["parquet"],
    )

    ## Get rooftop stats
    colony_rooftops["rooftop_area_m2"].hist(bins=50)
    plt.xlabel("Rooftop area (m2)")
    plt.ylabel("Number of rooftops")
    plt.title(f"{colony} Rooftop Area Distribution")
    plt.savefig(folder_path / "edited_rooftop_area_distribution.png")
    plt.close()

### Pivot to get colony-per-row dataset

In [ ]:
pivot_df = edited_colony_rooftops_gdf.pivot_table(index='Name', columns='rooftop_area_category', aggfunc='size', fill_value=0)
pivot_df = pivot_df.reset_index()
pivot_df

In [44]:
# CANCELLED # double the counts for Eco Green City and Crystal Green Welfare Society
# pivot_df.loc[
#     pivot_df["Name"].isin(["Eco Green City", "Crystal Green Welfare Society"]),
#     pivot_df.columns[1:],
# ] *= 2

In [ ]:
totals_df = (
    edited_colony_rooftops_gdf.groupby("Name")
    .agg(
        total_rooftop_area_m2=("rooftop_area_m2", "sum"),
        total_rooftop_count=("Name", "size"),
    )
    .reset_index()
)

In [ ]:
totals_df = pivot_df.merge(totals_df, on="Name")

## 5. Incorporate solar datasets

In [60]:
import rasterio
from rasterio.plot import show
from rasterstats import zonal_stats

In [61]:
solar_data_folderpath = (
    RAW_DATA_DIR
    / "solar_atlas"
    / "India_GISdata_LTAy_YearlyMonthlyTotals_GlobalSolarAtlas-v2_GEOTIFF"
)

### GTI, DIF, DNI, GHI

In [62]:
solar_irradiation_filenames = [
    "GTI.tif",
    "DIF.tif",
    "DNI.tif",
    "GHI.tif",
]
solar_irradiation_column_names = [
    "total_yearly_optimum_tilt_irradiation_kwh",
    "total_yearly_diffuse_irradiation_kwh",
    "total_yearly_direct_normal_irradiation_kwh",
    "total_yearly_horizontal_irradiation_kwh",
]

In [68]:
solar_stats = []
for filename, column_name in zip(solar_irradiation_filenames, solar_irradiation_column_names):

    raster = rasterio.open(solar_data_folderpath / filename)

    # # Plot
    # fig, ax = plt.subplots(figsize=(10, 10))

    # show(raster, ax=ax, title="GeoTIFF Raster")
    # subset_rooftops_gdf.plot(ax=ax, color="r")

    # # restrict the extent of the plot to the extent of the shapes
    # # ax.set_xlim(subset_rooftops_gdf.total_bounds[[0, 2]])
    # # ax.set_ylim(subset_rooftops_gdf.total_bounds[[1, 3]])
    # plt.show()

    # Prep data
    array = raster.read(1)
    affine = raster.transform
    # get per-rooftop irradiation
    solar_stats = zonal_stats(edited_colony_rooftops_gdf, array, affine=affine, nodata=raster.nodata, stats=['sum'], all_touched=True)
    # add to rooftops dataset
    edited_colony_rooftops_gdf[column_name] = [d["sum"] for d in solar_stats]
    # get per-colony stat
    irradiation_colony_totals = edited_colony_rooftops_gdf.groupby("Name")[column_name].sum()
    totals_df[column_name] = totals_df["Name"].map(irradiation_colony_totals)

### Add other data

Total yearly average of potential photovoltaic electricity production (PVOUT) in colony

Optimum tilt to maximaze yearly PV production in colony

Total yearly average of air temperature (TEMP) in colony

In [81]:
filename = "OPTA.tif"
column_name = "optimum_tilt_deg"

raster = rasterio.open(solar_data_folderpath / filename)
# Prep data
array = raster.read(1)
affine = raster.transform
# get per-rooftop irradiation
solar_stats = zonal_stats(
    edited_colony_rooftops_gdf,
    array,
    affine=affine,
    nodata=raster.nodata,
    stats=[
        "median",
    ],
    all_touched=True,
)
# add to rooftops dataset
edited_colony_rooftops_gdf[column_name] = [d["median"] for d in solar_stats]
# get per-colony stat
irradiation_colony_totals = edited_colony_rooftops_gdf.groupby("Name")[
    column_name
].median()
totals_df[column_name] = totals_df["Name"].map(irradiation_colony_totals)

In [83]:
filename = "TEMP.tif"
column_name = "avg_yearly_temp_c"

raster = rasterio.open(solar_data_folderpath / filename)
# Prep data
array = raster.read(1)
affine = raster.transform
# get per-rooftop irradiation
solar_stats = zonal_stats(
    edited_colony_rooftops_gdf,
    array,
    affine=affine,
    nodata=raster.nodata,
    stats=[
        "median",
    ],
    all_touched=True,
)
# add to rooftops dataset
edited_colony_rooftops_gdf[column_name] = [d["median"] for d in solar_stats]
# get per-colony stat
irradiation_colony_totals = edited_colony_rooftops_gdf.groupby("Name")[
    column_name
].median()
totals_df[column_name] = totals_df["Name"].map(irradiation_colony_totals)

In [ ]:
totals_df

### Finalise and save to file

In [85]:
totals_df = totals_df.round(0)

In [ ]:
totals_df

### 

In [72]:
totals_df.to_csv(PROCESSED_DATA_DIR / "edited_colony_stats.csv", index=False)